# Step 3 — Model Inference & Evaluation

**Research question:** Can any existing pretrained PhaseNet weight serve as a
competent *global* phase picker across local, regional, and teleseismic distances?

**Approach:** Run all PhaseNet pretrained weights through `benchmark_waveforms.hdf5`
and evaluate on three tasks following Münchmeyer et al. (2022):

| Task | Paper metric | Our metric | Notes |
|------|-------------|------------|-------|
| 1 — Detection | ROC-AUC (with noise windows) | Recall @ threshold + median probability | Event-only benchmark → no FPR; AUC requires noise windows |
| 2 — Phase ID | MCC | MCC | Traces with both P and S in window |
| 3 — Onset time | MAE, RMSE, outlier % (±1.5 s) | MAE, RMSE, outlier % (±1.5 s) | Exact match to paper |

**In-domain vs cross-domain:** Every result is reported both ways.
The `trained_models` column in the manifest identifies which dataset each trace
belongs to — when evaluating `PhaseNet/stead`, STEAD traces are excluded from
the cross-domain split. The MLAAPDE teleseismic traces are 100% cross-domain
for every model (none was trained on MLAAPDE).

**Key difference from Münchmeyer et al. (2022):**
The paper evaluates 6 model families (EQTransformer, GPD, PhaseNet, DDP, CRED, BasicPhaseAE)
on 8 fixed datasets. This notebook evaluates 15 PhaseNet weight variants on a 11-dataset
benchmark spanning local, regional, and teleseismic distances. Detection AUC requires paired
noise windows; we report recall@0.3 and median probability instead.

```
benchmark_waveforms.hdf5
        │
        ▼
 3.1  Setup & model inventory
        │
        ▼
 3.2  Inference loop
        │   For each PhaseNet weight:
        │     load model → batched GPU forward pass → extract P/S peaks → save
        ▼
 3.3  Metric computation (Münchmeyer et al. 2022 tasks)
        │   Task 1: Detection recall / median prob
        │   Task 2: Phase ID MCC
        │   Task 3: Onset MAE / RMSE / outlier fraction (±1.5 s)
        │   Broken down by: model · split (all / cross-domain) · distance bin
        ▼
 3.4  Summary table + visualisation
        ▼
     step3_results.parquet  +  step3_metrics.csv
```

## 3.1  Imports & Configuration


In [23]:
import numpy as np
import pandas as pd
import h5py, warnings
from pathlib import Path
from tqdm.notebook import tqdm
from collections import defaultdict
import scipy.signal
import torch
import seisbench.models as sbm

warnings.filterwarnings("ignore", category=UserWarning)

# ── Paths ──────────────────────────────────────────────────────────────────
HDF5_PATH    = Path("benchmark_waveforms.hdf5")
INDEX_PATH   = Path("benchmark_waveforms_index.csv")
RESULTS_PATH = Path("step3_results.parquet")
METRICS_PATH = Path("step3_metrics.csv")

assert HDF5_PATH.exists(),  "benchmark_waveforms.hdf5 not found — run Step 2 first"
assert INDEX_PATH.exists(), "benchmark_waveforms_index.csv not found — run Step 2 first"

# ── Inference parameters ───────────────────────────────────────────────────
BATCH_SIZE    = 64       # traces per GPU/CPU batch
TARGET_SR     = 100      # Hz
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD_P   = 0.3      # probability threshold for P detection recall
THRESHOLD_S   = 0.3      # probability threshold for S detection recall

# ── Outlier threshold — Münchmeyer et al. (2022) ──────────────────────────
# The paper defines outliers as |residual| > 1.5 s uniformly for all distances.
# We use one constant to match this definition exactly.
OUTLIER_THR_S = 1.50     # seconds — same threshold for local, regional, teleseismic

print(f"Device          : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Threshold P/S   : {THRESHOLD_P} / {THRESHOLD_S}")
print(f"Outlier thr (s) : {OUTLIER_THR_S}  (Münchmeyer et al. 2022 uniform definition)")

Device          : cuda
GPU             : NVIDIA GeForce RTX 3090
Batch size      : 64
Threshold P/S   : 0.3 / 0.3
Outlier thr (s) : 1.5  (Münchmeyer et al. 2022 uniform definition)


## 3.1b  Model Inventory

All 16 PhaseNet pretrained weights available in SeisBench, grouped by tier
from Step 1's model tiering analysis.

**Tier A (Generalist candidates):** `stead`, `instance`, `neic`
**Tier B (Specialty/large-regional):** `diting`, `obs`, `volpick`, `pisdl`, `phasenet_sn`, `jma`, `jma_wc`
**Tier C (Regional baselines):** `scedc`, `ethz`, `iquique`, `lendb`, `original`
**Tier D (Teleseismic):** `geofon`

The `trained_models` column in the manifest maps each trace to the weight(s)
that were trained on its source dataset — used for in-domain/cross-domain splits.

**Benchmark datasets (Step 2 output):**

| Dataset | In benchmark | Model trained on it |
|---------|-------------|---------------------|
| stead | yes | `stead`, `original` |
| instancecounts | yes | `instance` |
| pnw | yes | — |
| txed | yes | — |
| mlaapde | yes | — |
| ethz | yes | `ethz` |
| pisdl | yes | `pisdl` |
| ceed | yes | — |
| vcseis | yes | — |
| aq2009gm | yes | — |
| cwa | yes | — |
| obst2024 | skipped | `obs` |
| scedc | not in benchmark | `scedc` (→ all traces cross-domain) |
| iquique | not in benchmark | `iquique` (→ all traces cross-domain) |

In [24]:
# PhaseNet weights to evaluate — ordered by tier
PHASENET_WEIGHTS = {
    # Tier A — Generalist candidates
    "stead":       {"tier": "A", "trained_on": "stead",          "description": "Northern California (STEAD)"},
    "instance":    {"tier": "A", "trained_on": "instance",        "description": "Italy/Mediterranean (INSTANCE)"},
    "neic":        {"tier": "A", "trained_on": "neic",            "description": "Global teleseismic (NEIC)"},
    # Tier B — Specialty / large-regional
    "diting":      {"tier": "B", "trained_on": None,              "description": "China (~2.7M events)"},
    "obs":         {"tier": "B", "trained_on": "obst2024",        "description": "Ocean-bottom seismometers"},
    "volpick":     {"tier": "B", "trained_on": None,              "description": "Volcano-tectonic events"},
    "pisdl":       {"tier": "B", "trained_on": "pisdl",           "description": "Induced seismicity"},
    "phasenet_sn": {"tier": "B", "trained_on": None,              "description": "SNet offshore Japan"},
    "jma":         {"tier": "B", "trained_on": None,              "description": "JMA Japan"},
    "jma_wc":      {"tier": "B", "trained_on": None,              "description": "JMA Japan (wider conv)"},
    # Tier C — Regional baselines
    "scedc":       {"tier": "C", "trained_on": "scedc",           "description": "Southern California (SCEDC)"},
    "ethz":        {"tier": "C", "trained_on": "ethz",            "description": "Switzerland (ETHZ)"},
    "iquique":     {"tier": "C", "trained_on": "iquique",         "description": "Northern Chile (Iquique)"},
    "lendb":       {"tier": "C", "trained_on": None,              "description": "LenDB (Italy, P-only)"},
    "original":    {"tier": "C", "trained_on": "stead",           "description": "Original PhaseNet (Zhu & Beroza 2018)"},
    # Tier D — Teleseismic
    "geofon":      {"tier": "D", "trained_on": None,              "description": "Teleseismic broadband (GEOFON)"},
}

print(f"Total weights to evaluate: {len(PHASENET_WEIGHTS)}")
for tier in ["A","B","C","D"]:
    ws = [w for w,v in PHASENET_WEIGHTS.items() if v["tier"]==tier]
    print(f"  Tier {tier}: {ws}")

# Verify all weights are available in SeisBench
print("\nVerifying availability...")
available, missing = [], []
for weight in PHASENET_WEIGHTS:
    try:
        sbm.PhaseNet.from_pretrained(weight, update=False)
        available.append(weight)
    except Exception as e:
        missing.append((weight, str(e)[:60]))

print(f"  Available: {len(available)}")
if missing:
    print(f"  Missing  : {len(missing)}")
    for w, e in missing:
        print(f"    {w}: {e}")

Total weights to evaluate: 16
  Tier A: ['stead', 'instance', 'neic']
  Tier B: ['diting', 'obs', 'volpick', 'pisdl', 'phasenet_sn', 'jma', 'jma_wc']
  Tier C: ['scedc', 'ethz', 'iquique', 'lendb', 'original']
  Tier D: ['geofon']

Verifying availability...
  Available: 15
  Missing  : 1
    pisdl: Weights require seisbench version at least 0.10.1, but the i


## 3.2  Load Benchmark


In [25]:
# ── Load index and manifest ────────────────────────────────────────────────
index = pd.read_csv(INDEX_PATH)
ok    = index[index["status"] == "ok"].copy().reset_index(drop=True)

manifest = pd.read_csv("benchmark_manifest.csv")

# Merge physics columns — drop any existing source_month first to avoid
# duplicate columns from repeated runs
ok = ok.drop(columns=[c for c in ok.columns if "source_month" in c], errors="ignore")

phys = ["trace_name","magnitude","depth_km","distance_km","ts_tp_s",
        "dist_bin","depth_bin","mag_bin","trained_models","tectonic_type",
        "p_arrival_sample","s_arrival_sample","source_month","dataset"]
ok = ok.merge(
    manifest[[c for c in phys if c in manifest.columns]],
    on="trace_name", how="left", suffixes=("","_m"))

# Drop any _m duplicate columns created by the merge
ok = ok.drop(columns=[c for c in ok.columns if c.endswith("_m")], errors="ignore")

print(f"Benchmark traces: {len(ok):,}")
print(ok.groupby("dataset")["trace_name"].count().rename("n").to_string())

print(f"\nColumn NaN check:")
for col in ["p_arrival_sample","s_arrival_sample","source_month","dist_bin"]:
    if col in ok.columns:
        n_nan = ok[col].isna().sum()
        print(f"  {col:25s}: {n_nan:,} NaN  "
              f"({'expected — S-only or non-MLAAPDE' if n_nan > 0 else 'clean'})")

print(f"\nDistance bins:")
print(ok["dist_bin"].value_counts().to_string())

# ── Sanity check: the 1 NaN p_arrival_sample is the S-only MLAAPDE trace ──
bad = ok[ok["p_arrival_sample"].isna()]
if len(bad) > 0:
    print(f"\nTraces with no P arrival ({len(bad)}):")
    print(bad[["trace_name","dataset",
               "has_p_pick","has_s_pick"]].to_string(index=False))


Benchmark traces: 32,144
dataset
aq2009gm          1363
ceed              4530
cwa                195
ethz              1365
instancecounts    8406
mlaapde           1444
pisdl              762
pnw               2333
stead             9366
txed              2267
vcseis             113

Column NaN check:
  p_arrival_sample         : 43 NaN  (expected — S-only or non-MLAAPDE)
  s_arrival_sample         : 2,338 NaN  (expected — S-only or non-MLAAPDE)
  source_month             : 30,700 NaN  (expected — S-only or non-MLAAPDE)
  dist_bin                 : 308 NaN  (expected — S-only or non-MLAAPDE)

Distance bins:
dist_bin
local (<150km)           15915
regional (150-1500km)    14477
teleseismic (>1500km)     1444

Traces with no P arrival (43):
           trace_name  dataset  has_p_pick  has_s_pick
 bucket0$846,:3,:8751 aq2009gm       False        True
bucket13$347,:3,:8751 aq2009gm       False        True
bucket12$874,:3,:8751 aq2009gm       False        True
bucket17$985,:3,:8751 aq2009g

## 3.3  Inference Loop

Reads preprocessed 3000-sample / 100 Hz windows from `benchmark_waveforms.hdf5`
(produced by Step 2). The HDF5 is opened once and kept open for all model weights.

**Why this is fast vs the old `annotate()` approach:**
- Single flat HDF5 file — no per-trace file opens across 11 source datasets/shards
- `model.forward()` called once per batch of 64 traces on the RTX 3090
- No sliding-window or blinding overhead — windows are exactly 3000 samples

**Normalization:** Each batch is demeaned + unit-std per component before the
forward pass, matching what `annotate()` applies internally. The step-2 windows
are already in this form, so this is a near no-op that guards against edge cases.

**Peak search:** The output probability trace `pred[b, 0, :]` (P) and
`pred[b, 1, :]` (S) are searched within ±5 s of the known `p_in_window` /
`s_in_window` positions from the index. Residual = `(peak − true) / SR` in seconds.

**`in_samples` padding:** PhaseNet's canonical `in_samples = 3001`; the stored
windows are 3000 samples. The batch is zero-padded to 3001 before the forward
pass and output is trimmed back to 3000 — no effect on peak positions.

In [26]:
SEARCH_WINDOW_S = 5.0   # seconds either side of known arrival to search for peak


def _normalize_batch(batch):
    """Per-component demean + unit-std.  batch: (B, C, T) → (B, C, T) float32."""
    b   = batch - batch.mean(axis=-1, keepdims=True)
    std = b.std(axis=-1, keepdims=True)
    std[std < 1e-10] = 1.0
    return (b / std).astype(np.float32)


def _get_model_in_channels(model):
    """Detect expected input channels by inspecting the first Conv1d layer."""
    for module in model.modules():
        if isinstance(module, torch.nn.Conv1d):
            return module.in_channels
    return 3   # default


def run_inference_batched(weight_name, ok_df, hf):
    """
    Batched GPU inference on benchmark_waveforms.hdf5.

    Reads 3000-sample / 100 Hz windows (preprocessed by Step 2), applies
    per-component normalization, stacks into GPU batches of BATCH_SIZE, and
    calls model.forward() once per batch.

    Channel handling: most PhaseNet weights expect 3 channels (Z, N, E).
    The `obs` model expects 4 (adds hydrophone). When in_channels > 3, a
    zero-padded 4th channel is appended so the model can still run on
    3-component land data — this is a cross-domain evaluation.

    p_in_window / s_in_window from the index give exact arrival positions;
    the P/S peaks are searched within ±5 s of those positions.

    Degenerate-output flag: models whose median max-P-probability across the
    full window equals their max within the search region likely output flat
    probability traces (no localized peak). These are flagged in the results.
    """
    model = sbm.PhaseNet.from_pretrained(weight_name, update=False)
    model.eval()
    model.to(DEVICE)

    n_in     = int(getattr(model, "in_samples", 3001))
    n_chan   = _get_model_in_channels(model)
    SEARCH   = int(SEARCH_WINDOW_S * TARGET_SR)   # 500 samples

    results  = {}
    rows     = [(idx, row) for idx, row in ok_df.iterrows()]

    for start in range(0, len(rows), BATCH_SIZE):
        batch_rows  = rows[start : start + BATCH_SIZE]
        waves, meta = [], []

        for idx, row in batch_rows:
            tname = row["trace_name"]
            if tname not in hf["waveforms"]:
                results[idx] = {"error": "not_in_hdf5"}
                continue
            waves.append(hf["waveforms"][tname][:])   # (3, 3000) float32
            meta.append((idx, row))

        if not waves:
            continue

        batch_np = _normalize_batch(np.stack(waves))   # (B, 3, 3000)
        wave_len = batch_np.shape[-1]

        # Pad channel dim if model expects more than 3 components (e.g. obs: 4)
        if n_chan > batch_np.shape[1]:
            pad_ch   = n_chan - batch_np.shape[1]
            batch_np = np.pad(batch_np, ((0, 0), (0, pad_ch), (0, 0)))

        # Pad time dim to model's expected in_samples (3000 → 3001 for most)
        if wave_len < n_in:
            batch_np = np.pad(batch_np, ((0, 0), (0, 0), (0, n_in - wave_len)))

        batch_t = torch.tensor(batch_np, dtype=torch.float32).to(DEVICE)

        with torch.no_grad():
            out = model(batch_t)

        # Unpack output: tensor (B, C, T) or tuple (p_tensor, s_tensor)
        if isinstance(out, (tuple, list)):
            p_full = out[0].cpu().numpy()
            s_full = out[1].cpu().numpy()
        else:
            out_np = out.cpu().numpy()   # (B, C, T) — C[0]=P, C[1]=S
            p_full = out_np[:, 0, :]
            s_full = out_np[:, 1, :]

        # Trim back to original 3000 samples
        p_full = p_full[:, :wave_len]
        s_full = s_full[:, :wave_len]

        for i, (idx, row) in enumerate(meta):
            p_in = int(row["p_in_window"])
            s_in = int(row["s_in_window"])

            # ── P peak ──────────────────────────────────────────────────
            p_prob = 0.0;  p_res = np.nan
            if p_in >= 0:
                ps = max(0, p_in - SEARCH);  pe = min(wave_len, p_in + SEARCH)
                pk = int(np.argmax(p_full[i, ps:pe])) + ps
                p_prob = float(p_full[i, pk])
                p_res  = (pk - p_in) / TARGET_SR

            # ── S peak ──────────────────────────────────────────────────
            s_prob = 0.0;  s_res = np.nan
            if s_in >= 0:
                ss = max(0, s_in - SEARCH);  se = min(wave_len, s_in + SEARCH)
                sk = int(np.argmax(s_full[i, ss:se])) + ss
                s_prob = float(s_full[i, sk])
                s_res  = (sk - s_in) / TARGET_SR

            results[idx] = {
                "p_prob":  round(p_prob, 4),
                "p_res_s": round(float(p_res), 4) if not np.isnan(p_res) else np.nan,
                "s_prob":  round(s_prob, 4),
                "s_res_s": round(float(s_res), 4) if not np.isnan(s_res) else np.nan,
            }

    model.cpu()
    del model
    torch.cuda.empty_cache()
    return results


# ── Main inference loop ───────────────────────────────────────────────────
all_results = []

print(f"Running batched inference for {len(available)} weights "
      f"over {len(ok):,} traces  (batch={BATCH_SIZE}, device={DEVICE})...")

with h5py.File(HDF5_PATH, "r") as hf:
    for weight_name in tqdm(available, desc="Models"):
        print(f"  {weight_name}...", end=" ", flush=True)
        try:
            preds = run_inference_batched(weight_name, ok, hf)
        except Exception as e:
            print(f"FAILED: {e}")
            continue

        n_ok  = sum(1 for v in preds.values() if "error" not in v)
        n_err = len(preds) - n_ok
        print(f"{n_ok:,} ok  |  {n_err:,} errors")

        for idx, row in ok.iterrows():
            pred = preds.get(idx, {})
            if "error" in pred or not pred:
                continue
            all_results.append({
                "weight":         weight_name,
                "tier":           PHASENET_WEIGHTS[weight_name]["tier"],
                "trace_name":     row["trace_name"],
                "dataset":        row["dataset"],
                "dist_bin":       row.get("dist_bin",  np.nan),
                "depth_bin":      row.get("depth_bin", np.nan),
                "mag_bin":        row.get("mag_bin",   np.nan),
                "trained_models": str(row.get("trained_models", "")),
                "snr_db":         row.get("snr_db",    np.nan),
                "p_in_window":    int(row["p_in_window"]),
                "s_in_window":    int(row["s_in_window"]),
                "p_prob":         pred["p_prob"],
                "s_prob":         pred["s_prob"],
                "p_residual_s":   pred["p_res_s"],
                "s_residual_s":   pred["s_res_s"],
            })

results_df = pd.DataFrame(all_results)
results_df.to_parquet(RESULTS_PATH, index=False)
print(f"\nSaved {len(results_df):,} rows → {RESULTS_PATH}")

# ── Degenerate-output check ───────────────────────────────────────────────
# Models with recall ≈ 1.0 AND MAE >> 1 s are outputting flat probability
# traces — every position is above threshold, but the "peak" is just noise.
# These need annotation-based inference or a different normalization scheme.
print("\nDegenerate output check (recall=1.0 + MAE > 2s flags flat prob traces):")
for wname in results_df["weight"].unique():
    wdf = results_df[results_df["weight"] == wname]
    rec = (wdf["p_prob"] >= THRESHOLD_P).mean()
    mae = wdf["p_residual_s"].abs().mean()
    if rec > 0.99 and mae > 2.0:
        print(f"  ⚠  {wname:15s}: recall={rec:.3f}  P-MAE={mae:.2f}s  "
              f"→ flat output, results unreliable")

Running batched inference for 15 weights over 32,144 traces  (batch=64, device=cuda)...


Models:   0%|          | 0/15 [00:00<?, ?it/s]

  stead... 32,144 ok  |  0 errors
  instance... 32,144 ok  |  0 errors
  neic... 32,144 ok  |  0 errors
  diting... 32,144 ok  |  0 errors
  obs... 32,144 ok  |  0 errors
  volpick... 32,144 ok  |  0 errors
  phasenet_sn... 32,144 ok  |  0 errors
  jma... 32,144 ok  |  0 errors
  jma_wc... 32,144 ok  |  0 errors
  scedc... 32,144 ok  |  0 errors
  ethz... 32,144 ok  |  0 errors
  iquique... 32,144 ok  |  0 errors
  lendb... 32,144 ok  |  0 errors
  original... 32,144 ok  |  0 errors
  geofon... 32,144 ok  |  0 errors

Saved 482,160 rows → step3_results.parquet

Degenerate output check (recall=1.0 + MAE > 2s flags flat prob traces):
  ⚠  diting         : recall=0.999  P-MAE=3.72s  → flat output, results unreliable
  ⚠  phasenet_sn    : recall=0.999  P-MAE=3.68s  → flat output, results unreliable
  ⚠  original       : recall=0.999  P-MAE=3.09s  → flat output, results unreliable


## 3.4  Metric Computation  ·  Münchmeyer et al. (2022) task definitions

**Task 1 — Detection** (paper: ROC-AUC; our proxy: recall + median probability)
The paper computes AUC using event and noise windows. Our benchmark contains only
event windows, so true-positive rate (recall) and median detection probability are
reported instead. Recall@0.3 is the fraction of event traces where the model's peak
P (or S) probability exceeds 0.3. Median probability is threshold-free and gauges
overall detection confidence.

**Task 2 — Phase identification MCC** (paper: exact match)
At the true P arrival position, is P_prob > S_prob? At the true S arrival position,
is S_prob > P_prob? Matthews Correlation Coefficient on these binary outcomes,
restricted to traces where both P and S fall inside the 30 s window.

**Task 3 — Onset time picking** (paper: exact match)
MAE and RMSE of pick residuals (predicted − true) in seconds. Outlier fraction =
fraction of picks with |residual| > **1.5 s**, applied uniformly to all distance
bins following Münchmeyer et al. (2022). (Previous notebooks used 0.45 s for
regional — that was incorrect.)

Each metric is computed in two splits:
- **All** — all traces including in-domain ones
- **Cross-domain** — excluding traces from the model's own training dataset

In [27]:
from sklearn.metrics import matthews_corrcoef
results_df = pd.read_parquet(RESULTS_PATH)

# Mark models whose outputs are degenerate (flat probability → unreliable)
DEGENERATE_MODELS = set()
for wname in results_df["weight"].unique():
    wdf = results_df[results_df["weight"] == wname]
    rec = (wdf["p_prob"] >= THRESHOLD_P).mean()
    mae = wdf["p_residual_s"].abs().mean()
    if rec > 0.99 and mae > 2.0:
        DEGENERATE_MODELS.add(wname)

if DEGENERATE_MODELS:
    print(f"Degenerate models (flat output — excluded from metric summary): "
          f"{sorted(DEGENERATE_MODELS)}")


def compute_metrics(df, weight_name, split_name, dist_label=None):
    """
    Compute all three Münchmeyer et al. (2022) tasks.

    Task 1 — Detection
      p_recall / s_recall  : fraction of event traces with peak prob >= threshold
      p_med_prob           : median peak P probability (threshold-free detection quality)

    Task 2 — Phase identification
      mcc                  : Matthews Correlation Coefficient (P vs S at known positions)

    Task 3 — Onset time picking  (Münchmeyer et al. 2022 exact definitions)
      p_mae_s / s_mae_s    : mean absolute error in seconds
      p_rmse_s / s_rmse_s  : root mean squared error in seconds
      p_outlier / s_outlier: fraction with |residual| > OUTLIER_THR_S (1.5 s uniform)
    """
    if len(df) == 0:
        return None

    # ── Task 1: Detection ─────────────────────────────────────────────────
    p_traces = df[df["p_in_window"] >= 0]
    s_traces = df[df["s_in_window"] >= 0]

    p_recall   = (p_traces["p_prob"] >= THRESHOLD_P).mean() if len(p_traces) > 0 else np.nan
    s_recall   = (s_traces["s_prob"] >= THRESHOLD_S).mean() if len(s_traces) > 0 else np.nan
    p_med_prob = p_traces["p_prob"].median() if len(p_traces) > 0 else np.nan
    s_med_prob = s_traces["s_prob"].median() if len(s_traces) > 0 else np.nan

    # ── Task 2: Phase identification MCC ─────────────────────────────────
    both = df[(df["p_in_window"] >= 0) & (df["s_in_window"] >= 0)].copy()
    mcc  = np.nan
    if len(both) >= 5:
        p_correct = (both["p_prob"] > both["s_prob"]).astype(int)
        s_correct = (both["s_prob"] > both["p_prob"]).astype(int)
        y_true = np.concatenate([np.ones(len(both)),  np.zeros(len(both))])
        y_pred = np.concatenate([p_correct.values,    s_correct.values])
        try:
            mcc = matthews_corrcoef(y_true, y_pred)
        except Exception:
            mcc = np.nan

    # ── Task 3: Onset time picking ────────────────────────────────────────
    p_res = df.loc[df["p_in_window"] >= 0, "p_residual_s"].dropna()
    s_res = df.loc[df["s_in_window"] >= 0, "s_residual_s"].dropna()

    p_mae  = np.abs(p_res).mean()       if len(p_res) > 0 else np.nan
    p_rmse = np.sqrt((p_res**2).mean()) if len(p_res) > 0 else np.nan
    s_mae  = np.abs(s_res).mean()       if len(s_res) > 0 else np.nan
    s_rmse = np.sqrt((s_res**2).mean()) if len(s_res) > 0 else np.nan

    # Outlier fraction: uniform ±1.5 s (Münchmeyer et al. 2022, all distances)
    p_outlier = (np.abs(p_res) > OUTLIER_THR_S).mean() if len(p_res) > 0 else np.nan
    s_outlier = (np.abs(s_res) > OUTLIER_THR_S).mean() if len(s_res) > 0 else np.nan

    return {
        "weight":        weight_name,
        "tier":          PHASENET_WEIGHTS.get(weight_name, {}).get("tier", "?"),
        "split":         split_name,
        "dist_bin":      dist_label or "all",
        "n_traces":      len(df),
        "degenerate":    weight_name in DEGENERATE_MODELS,
        # Task 1
        "p_recall":      round(p_recall,   4) if not np.isnan(p_recall)   else np.nan,
        "s_recall":      round(s_recall,   4) if not np.isnan(s_recall)   else np.nan,
        "p_med_prob":    round(p_med_prob, 4) if not np.isnan(p_med_prob) else np.nan,
        "s_med_prob":    round(s_med_prob, 4) if not np.isnan(s_med_prob) else np.nan,
        # Task 2
        "mcc":           round(mcc,        4) if not np.isnan(mcc)        else np.nan,
        # Task 3
        "p_mae_s":       round(p_mae,      4) if not np.isnan(p_mae)      else np.nan,
        "p_rmse_s":      round(p_rmse,     4) if not np.isnan(p_rmse)     else np.nan,
        "s_mae_s":       round(s_mae,      4) if not np.isnan(s_mae)      else np.nan,
        "s_rmse_s":      round(s_rmse,     4) if not np.isnan(s_rmse)     else np.nan,
        "p_outlier":     round(p_outlier,  4) if not np.isnan(p_outlier)  else np.nan,
        "s_outlier":     round(s_outlier,  4) if not np.isnan(s_outlier)  else np.nan,
        "outlier_thr_s": OUTLIER_THR_S,
    }


metrics_rows = []
dist_bins    = results_df["dist_bin"].dropna().unique().tolist() + ["all"]

for weight_name in tqdm(results_df["weight"].unique(), desc="Computing metrics"):
    wdf        = results_df[results_df["weight"] == weight_name]
    trained_on = PHASENET_WEIGHTS.get(weight_name, {}).get("trained_on", None)

    # Cross-domain mask: exclude traces whose source dataset matches trained_on
    if trained_on:
        cross_mask = ~wdf["trained_models"].str.contains(
            trained_on, na=False, regex=False)
    else:
        cross_mask = pd.Series(True, index=wdf.index)

    for dist_label in dist_bins:
        if dist_label == "all":
            subset_all   = wdf
            subset_cross = wdf[cross_mask]
        else:
            subset_all   = wdf[wdf["dist_bin"] == dist_label]
            subset_cross = wdf[cross_mask & (wdf["dist_bin"] == dist_label)]

        row_all   = compute_metrics(subset_all,   weight_name, "all",          dist_label)
        row_cross = compute_metrics(subset_cross, weight_name, "cross_domain", dist_label)
        if row_all:   metrics_rows.append(row_all)
        if row_cross: metrics_rows.append(row_cross)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(METRICS_PATH, index=False)
print(f"Saved {len(metrics_df):,} metric rows → {METRICS_PATH}")
print(f"\nOutlier threshold : ±{OUTLIER_THR_S} s  (Münchmeyer et al. 2022, uniform)")
print(f"Degenerate models : {sorted(DEGENERATE_MODELS) or 'none'}\n")
print(metrics_df[metrics_df["split"] == "cross_domain"][
    metrics_df["dist_bin"] == "all"
].sort_values("p_mae_s")[
    ["weight","tier","degenerate","p_recall","p_med_prob","mcc",
     "p_mae_s","p_outlier","s_mae_s"]
].to_string(index=False))

Degenerate models (flat output — excluded from metric summary): ['diting', 'original', 'phasenet_sn']


Computing metrics:   0%|          | 0/15 [00:00<?, ?it/s]

Saved 119 metric rows → step3_metrics.csv

Outlier threshold : ±1.5 s  (Münchmeyer et al. 2022, uniform)
Degenerate models : ['diting', 'original', 'phasenet_sn']

     weight tier  degenerate  p_recall  p_med_prob     mcc  p_mae_s  p_outlier  s_mae_s
     jma_wc    B       False    0.8812      0.7806  0.7897   0.3742     0.0705   1.4227
        jma    B       False    0.8701      0.8082  0.7943   0.3814     0.0728   1.4330
      scedc    C       False    0.8345      0.7382  0.8721   0.5410     0.1145   1.4961
   instance    A       False    0.6040      0.4710  0.7440   0.6881     0.1459   1.7386
       neic    A       False    0.5898      0.3958  0.4950   0.7024     0.1526   1.5205
    volpick    B       False    0.8573      0.7612  0.6346   0.7915     0.1760   1.6117
        obs    B       False    0.6103      0.5654  0.5266   0.8909     0.1997   1.6614
       ethz    C       False    0.6386      0.5225  0.6835   1.0184     0.2394   1.7397
    iquique    C       False    0.3924      

## 3.5  Summary Table


In [28]:
metrics_df = pd.read_csv(METRICS_PATH)

# Exclude degenerate models from summary (flat probability output — results unreliable)
clean = metrics_df[~metrics_df["degenerate"].fillna(False)]

# ── Cross-domain summary across all distances ─────────────────────────────
summary = (clean[(clean["split"] == "cross_domain") & (clean["dist_bin"] == "all")]
           .sort_values(["tier","weight"])
           [["tier","weight","n_traces",
             "p_recall","p_med_prob",       # Task 1 — detection
             "mcc",                          # Task 2 — phase ID
             "p_mae_s","s_mae_s","p_outlier" # Task 3 — onset time
             ]]
           .set_index(["tier","weight"]))

print("Cross-domain performance — all distances  (outlier thr = ±1.5 s, Münchmeyer et al. 2022)")
print("Degenerate models (flat output) excluded from this table.")
print("=" * 90)
print(summary.to_string(float_format="{:.3f}".format))

# ── Per-distance breakdown for top 5 models by cross-domain P-MAE ────────
top5 = (clean[(clean["split"]=="cross_domain") & (clean["dist_bin"]=="all")]
        .nsmallest(5, "p_mae_s")["weight"].tolist())

print(f"\n\nTop 5 models by cross-domain P-MAE (lower = better): {top5}")
print("\nPer-distance P-MAE (s):")
dist_summary = (clean[(clean["split"]=="cross_domain") &
                       (clean["weight"].isin(top5)) &
                       (clean["dist_bin"] != "all")]
                .pivot_table(index="weight", columns="dist_bin",
                             values="p_mae_s", aggfunc="first")
                .round(3))
print(dist_summary.to_string())

if DEGENERATE_MODELS:
    print(f"\nExcluded degenerate models: {sorted(DEGENERATE_MODELS)}")
    print("(recall=1.0 + MAE>2s → model outputs flat probability — run annotate() "
          "or fix normalization to evaluate these.)")

Cross-domain performance — all distances  (outlier thr = ±1.5 s, Münchmeyer et al. 2022)
Degenerate models (flat output) excluded from this table.
               n_traces  p_recall  p_med_prob    mcc  p_mae_s  s_mae_s  p_outlier
tier weight                                                                      
A    instance     23738     0.604       0.471  0.744    0.688    1.739      0.146
     neic         30700     0.590       0.396  0.495    0.702    1.520      0.153
     stead        22778     0.522       0.371 -0.289    1.876    1.072      0.484
B    jma          32144     0.870       0.808  0.794    0.381    1.433      0.073
     jma_wc       32144     0.881       0.781  0.790    0.374    1.423      0.070
     obs          32144     0.610       0.565  0.527    0.891    1.661      0.200
     volpick      32144     0.857       0.761  0.635    0.791    1.612      0.176
C    ethz         30779     0.639       0.522  0.683    1.018    1.740      0.239
     iquique      32144     0.392

## 3.6  Visualisation  ·  Münchmeyer et al. (2022) figure style

Three figures following the paper's visual conventions. All use cross-domain results only. Degenerate models excluded.

**Figure A — P-residual histograms (paper Fig. 2 style)**
Grid: distance bin (row) × model (column). Blue bars. Corner box shows OUT / MAE / RMSE.
Reference lines: zero (grey), median (red dashed), mean (orange dashed).
X-range widens from local → regional → teleseismic.

**Figure B — S-residual histograms (paper Fig. 3 style)**
Same layout, orange bars. Wider X-range. Only traces with `s_in_window ≥ 0`.

**Figure C — Box-and-whisker summary (paper Fig. 5 style)**
One subplot per distance bin. Box = IQR (25th–75th %ile), solid line = median,
dashed line = mean, whiskers = 10th–90th %ile. Models on X-axis coloured by tier.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

results_df = pd.read_parquet(RESULTS_PATH)
metrics_df = pd.read_csv(METRICS_PATH)

# ── Model order: non-degenerate, sorted by tier then cross-domain P-MAE ──
mae_lookup = (metrics_df[(metrics_df["split"] == "cross_domain") &
                          (metrics_df["dist_bin"] == "all")]
              .set_index("weight")["p_mae_s"].to_dict())

eval_weights = sorted(
    [w for w in results_df["weight"].unique() if w not in DEGENERATE_MODELS],
    key=lambda w: (PHASENET_WEIGHTS.get(w, {}).get("tier", "Z"), mae_lookup.get(w, 99))
)

DIST_ORDER  = ["local (<150km)", "regional (150-1500km)", "teleseismic (>1500km)"]
DIST_LABELS = ["Local  (<150 km)", "Regional  (150–1500 km)", "Teleseismic  (>1500 km)"]
XLIMS       = [(-1.5,  1.5), (-2.0,  2.0), (-5.0,  5.0)]
BIN_WIDTHS  = [ 0.04,         0.06,         0.20         ]

# Cross-domain mask per weight
def cross_domain_mask(wdf, weight):
    trained_on = PHASENET_WEIGHTS.get(weight, {}).get("trained_on")
    if trained_on:
        return ~wdf["trained_models"].str.contains(trained_on, na=False, regex=False)
    return pd.Series(True, index=wdf.index)

n_rows, n_cols = len(DIST_ORDER), len(eval_weights)
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(2.6 * n_cols, 2.9 * n_rows),
                          constrained_layout=True)

fig.suptitle(r"P Pick Residuals — Cross-Domain   ($t_{\rm pred} - t_{\rm true}$, s)",
             fontsize=14, fontweight="bold")

for ri, (dist_bin, dist_lbl, (xlo, xhi), bw) in enumerate(
        zip(DIST_ORDER, DIST_LABELS, XLIMS, BIN_WIDTHS)):

    bins = np.arange(xlo - bw / 2, xhi + bw, bw)

    for ci, weight in enumerate(eval_weights):
        ax = axes[ri, ci]

        wdf  = results_df[results_df["weight"] == weight]
        mask = cross_domain_mask(wdf, weight)
        sub  = wdf[mask & (wdf["dist_bin"] == dist_bin) &
                    (wdf["p_in_window"] >= 0) &
                    wdf["p_residual_s"].notna()]

        if len(sub) < 5:
            ax.set_facecolor("#ebebeb")
            ax.text(0.5, 0.5, "n/a", ha="center", va="center",
                    transform=ax.transAxes, fontsize=9, color="#888")
        else:
            res      = sub["p_residual_s"].values
            out_frac = (np.abs(res) > OUTLIER_THR_S).mean()
            mae      = np.abs(res).mean()
            rmse     = np.sqrt((res ** 2).mean())
            med      = float(np.median(res))
            mn       = float(res.mean())

            ax.hist(res, bins=bins, color="#4472C4", alpha=0.88,
                    edgecolor="white", linewidth=0.2)

            ax.axvline(0,   color="#888888",    lw=0.8, zorder=3)
            ax.axvline(med, color="red",         lw=1.3, linestyle="--", zorder=4)
            ax.axvline(mn,  color="darkorange",  lw=1.3, linestyle="--", zorder=4)

            stats = f"OUT  {out_frac:.2f}\nMAE  {mae:.2f} s\nRMSE {rmse:.2f} s"
            ax.text(0.03, 0.97, stats, transform=ax.transAxes,
                    fontsize=6.5, va="top", ha="left", linespacing=1.5,
                    family="monospace",
                    bbox=dict(boxstyle="round,pad=0.25", fc="white",
                              ec="#aaa", alpha=0.92, lw=0.5))
            ax.set_facecolor("white")

        ax.set_xlim(xlo, xhi)
        ax.set_yticks([])
        ax.tick_params(axis="x", labelsize=7, pad=2)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4, prune="both"))
        ax.spines[["top", "right", "left"]].set_visible(False)

        if ri == 0:
            tier = PHASENET_WEIGHTS.get(weight, {}).get("tier", "?")
            ax.set_title(f"{weight}\n(T{tier})", fontsize=8.5,
                         fontweight="bold", pad=3)
        if ci == 0:
            ax.set_ylabel(dist_lbl, fontsize=9, fontweight="bold", labelpad=4)
        if ri == n_rows - 1:
            ax.set_xlabel(r"$t_{\rm pred} - t_{\rm true}$ (s)", fontsize=7.5)

plt.savefig("step3_p_residual_histograms.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.show()
print("Saved → step3_p_residual_histograms.png")

In [ ]:
# ── Figure B: S-residual histograms (paper Fig. 3 style) ─────────────────
XLIMS_S     = [(-2.0,  2.0), (-3.0,  3.0), (-7.0,  7.0)]
BIN_WIDTHS_S = [ 0.06,         0.10,         0.30         ]

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(2.6 * n_cols, 2.9 * n_rows),
                          constrained_layout=True)

fig.suptitle(r"S Pick Residuals — Cross-Domain   ($t_{\rm pred} - t_{\rm true}$, s)",
             fontsize=14, fontweight="bold")

for ri, (dist_bin, dist_lbl, (xlo, xhi), bw) in enumerate(
        zip(DIST_ORDER, DIST_LABELS, XLIMS_S, BIN_WIDTHS_S)):

    bins = np.arange(xlo - bw / 2, xhi + bw, bw)

    for ci, weight in enumerate(eval_weights):
        ax = axes[ri, ci]

        wdf  = results_df[results_df["weight"] == weight]
        mask = cross_domain_mask(wdf, weight)
        sub  = wdf[mask & (wdf["dist_bin"] == dist_bin) &
                    (wdf["s_in_window"] >= 0) &
                    wdf["s_residual_s"].notna()]

        if len(sub) < 5:
            ax.set_facecolor("#ebebeb")
            ax.text(0.5, 0.5, "n/a", ha="center", va="center",
                    transform=ax.transAxes, fontsize=9, color="#888")
        else:
            res      = sub["s_residual_s"].values
            out_frac = (np.abs(res) > OUTLIER_THR_S).mean()
            mae      = np.abs(res).mean()
            rmse     = np.sqrt((res ** 2).mean())
            med      = float(np.median(res))
            mn       = float(res.mean())

            ax.hist(res, bins=bins, color="#ED7D31", alpha=0.88,
                    edgecolor="white", linewidth=0.2)

            ax.axvline(0,   color="#888888",   lw=0.8, zorder=3)
            ax.axvline(med, color="red",        lw=1.3, linestyle="--", zorder=4)
            ax.axvline(mn,  color="#0070C0",    lw=1.3, linestyle="--", zorder=4)

            stats = f"OUT  {out_frac:.2f}\nMAE  {mae:.2f} s\nRMSE {rmse:.2f} s"
            ax.text(0.03, 0.97, stats, transform=ax.transAxes,
                    fontsize=6.5, va="top", ha="left", linespacing=1.5,
                    family="monospace",
                    bbox=dict(boxstyle="round,pad=0.25", fc="white",
                              ec="#aaa", alpha=0.92, lw=0.5))
            ax.set_facecolor("white")

        ax.set_xlim(xlo, xhi)
        ax.set_yticks([])
        ax.tick_params(axis="x", labelsize=7, pad=2)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4, prune="both"))
        ax.spines[["top", "right", "left"]].set_visible(False)

        if ri == 0:
            tier = PHASENET_WEIGHTS.get(weight, {}).get("tier", "?")
            ax.set_title(f"{weight}\n(T{tier})", fontsize=8.5,
                         fontweight="bold", pad=3)
        if ci == 0:
            ax.set_ylabel(dist_lbl, fontsize=9, fontweight="bold", labelpad=4)
        if ri == n_rows - 1:
            ax.set_xlabel(r"$t_{\rm pred} - t_{\rm true}$ (s)", fontsize=7.5)

plt.savefig("step3_s_residual_histograms.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.show()
print("Saved → step3_s_residual_histograms.png")

# ── Figure C: Box-and-whisker summary (paper Fig. 5 style) ───────────────
# Box = IQR (25–75 %ile)  ·  solid line = median  ·  dashed line = mean
# Whiskers = 10th–90th percentile  (as stated in Münchmeyer et al. 2022 caption)

TIER_COLORS = {"A": "#4472C4", "B": "#ED7D31", "C": "#70AD47", "D": "#9C27B0"}

fig, axes = plt.subplots(1, n_rows,
                          figsize=(5.0 * n_rows, 5.5),
                          constrained_layout=True)

fig.suptitle("P Pick Residuals — Cross-Domain  ·  Box-and-Whisker Summary\n"
             "(box = IQR, line = median, dashed = mean, whiskers = 10th–90th %ile)",
             fontsize=12, fontweight="bold")

for ax, dist_bin, dist_lbl in zip(axes, DIST_ORDER, DIST_LABELS):
    all_res, positions, colors, tick_labels = [], [], [], []

    for ci, weight in enumerate(eval_weights):
        wdf  = results_df[results_df["weight"] == weight]
        mask = cross_domain_mask(wdf, weight)
        sub  = wdf[mask & (wdf["dist_bin"] == dist_bin) &
                    (wdf["p_in_window"] >= 0) &
                    wdf["p_residual_s"].notna()]

        res = sub["p_residual_s"].values if len(sub) >= 5 else np.array([np.nan])
        all_res.append(res)
        positions.append(ci)
        tier = PHASENET_WEIGHTS.get(weight, {}).get("tier", "?")
        colors.append(TIER_COLORS.get(tier, "#888"))
        tick_labels.append(weight)

    # Draw each box individually to apply per-model colour
    for ci, (res, col) in enumerate(zip(all_res, colors)):
        if np.all(np.isnan(res)) or len(res) < 5:
            continue
        res = res[~np.isnan(res)]

        q10, q25, q50, q75, q90 = np.percentile(res, [10, 25, 50, 75, 90])
        mn = res.mean()

        # Box
        box = plt.matplotlib.patches.FancyBboxPatch(
            (ci - 0.35, q25), 0.70, q75 - q25,
            boxstyle="square,pad=0", linewidth=1.0,
            edgecolor="black", facecolor=col, alpha=0.55, zorder=2)
        ax.add_patch(box)

        # Median line
        ax.plot([ci - 0.35, ci + 0.35], [q50, q50],
                color="black", lw=1.8, zorder=3)
        # Mean dashed line
        ax.plot([ci - 0.35, ci + 0.35], [mn, mn],
                color="black", lw=1.2, linestyle="--", zorder=3)
        # Whiskers (10th–90th)
        ax.plot([ci, ci], [q10, q25], color="black", lw=1.0, zorder=2)
        ax.plot([ci, ci], [q75, q90], color="black", lw=1.0, zorder=2)
        ax.plot([ci - 0.2, ci + 0.2], [q10, q10], color="black", lw=1.0)
        ax.plot([ci - 0.2, ci + 0.2], [q90, q90], color="black", lw=1.0)

    ax.axhline(0, color="#888", lw=0.8, linestyle=":")
    ax.set_xlim(-0.7, n_cols - 0.3)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(tick_labels, rotation=40, ha="right", fontsize=9)
    ax.set_ylabel(r"$t_{\rm pred} - t_{\rm true}$ (s)", fontsize=10)
    ax.set_title(dist_lbl, fontsize=11, fontweight="bold", pad=6)
    ax.set_facecolor("white")
    ax.yaxis.grid(True, alpha=0.3, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    # Tier legend on first panel only
    if ax is axes[0]:
        handles = [plt.matplotlib.patches.Patch(fc=c, ec="black", alpha=0.6, label=f"Tier {t}")
                   for t, c in TIER_COLORS.items()]
        ax.legend(handles=handles, fontsize=8, loc="upper right",
                  framealpha=0.85, edgecolor="#ccc")

plt.savefig("step3_boxwhisker.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved → step3_boxwhisker.png")

## 3.7  Thesis-Ready Results Table


In [31]:
metrics_df = pd.read_csv(METRICS_PATH)
clean      = metrics_df[~metrics_df["degenerate"].fillna(False)]

# ── Full cross-domain results table: per distance bin ─────────────────────
# Columns: p_mae_s (Task 3 — onset MAE) | p_recall (Task 1) | mcc (Task 2)
# Outlier threshold: ±1.5 s uniform (Münchmeyer et al. 2022)
table = (clean[clean["split"] == "cross_domain"]
         .pivot_table(index=["tier","weight"],
                      columns="dist_bin",
                      values=["p_mae_s","p_recall","mcc"],
                      aggfunc="first")
         .round(3))

print("Cross-domain results by distance bin")
print("Metrics: p_mae_s = P onset MAE (s) | p_recall = detection recall@0.3 | mcc = phase ID")
print(f"Outlier threshold: ±{OUTLIER_THR_S} s (Münchmeyer et al. 2022 uniform)")
print("Degenerate models excluded:", sorted(DEGENERATE_MODELS) or "none")
print("=" * 110)
print(table.to_string())

# ── Teleseismic gap — key thesis finding ─────────────────────────────────
tele = (clean[(clean["split"]=="cross_domain") &
              (clean["dist_bin"]=="teleseismic (>1500km)")]
        .sort_values("p_mae_s")
        [["weight","tier","n_traces","p_recall","p_med_prob","p_mae_s","p_outlier","mcc"]])

print("\n\nTeleseismic cross-domain performance (sorted by P-MAE):")
print(tele.to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────
table.to_csv("step3_thesis_table.csv")
tele.to_csv("step3_teleseismic_table.csv", index=False)
print("\nSaved step3_thesis_table.csv and step3_teleseismic_table.csv")

Cross-domain results by distance bin
Metrics: p_mae_s = P onset MAE (s) | p_recall = detection recall@0.3 | mcc = phase ID
Outlier threshold: ±1.5 s (Münchmeyer et al. 2022 uniform)
Degenerate models excluded: ['diting', 'original', 'phasenet_sn']
                 mcc                                      p_mae_s                                                            p_recall                                                           
dist_bin         all local (<150km) regional (150-1500km)     all local (<150km) regional (150-1500km) teleseismic (>1500km)      all local (<150km) regional (150-1500km) teleseismic (>1500km)
tier weight                                                                                                                                                                                     
A    instance  0.744          0.713                 0.825   0.688          0.615                 0.570                 2.130    0.604          0.670                 0.590   